# L2 · El dataset de LeRobot por dentro

En el notebook 02 convertimos y miramos el resultado por arriba. Acá lo
desarmamos entero: por qué tiene esta forma, qué hay en cada archivo, todas
las formas de leerlo y cómo se edita.

- **Lo principal (1–5):** el diseño, el recorrido archivo por archivo,
  `LeRobotDataset`, ventanas de tiempo y aumentación de imágenes.
- **Para profundizar (6–7):** editar datasets (borrar episodios, partir,
  unir) y detalles de bajo nivel.

Usamos el dataset de entrenamiento de la primera ACT y el chiquito de 3
episodios que crea el notebook 02 (`nb_demo`; si no existe, corré antes el 02).

In [ ]:
import torch  # antes que lerobot: evita un import circular entre torchvision y torch._dynamo
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

from toolkit.paths import DATASETS_DIR, RAW_RECORDINGS_DIR

plt.rcParams["figure.dpi"] = 110
ROOT = DATASETS_DIR / "local" / "pick_mameluco_act_v1"
DEMO_ROOT = DATASETS_DIR / "local" / "nb_demo"
CAMERA = "observation.images.cam_right_high"

## 1. Por qué tiene esta forma

**La forma obvia** de guardar demostraciones es la de Unitree: una carpeta por
episodio, un JSON con los números, un JPEG por cuadro. Es fácil de escribir y
de leer a mano. ¿Dónde se rompe?

In [ ]:
raw_task = RAW_RECORDINGS_DIR / "pick_mameluco"
raw_files = list(raw_task.rglob("*"))
raw_bytes = sum(p.stat().st_size for p in raw_files if p.is_file())
converted = [p for p in ROOT.rglob("*") if p.is_file()]
converted_bytes = sum(p.stat().st_size for p in converted)
print(f"crudo    : {len(raw_files):6d} archivos · {raw_bytes / 1e9:.2f} GB")
print(f"LeRobot  : {len(converted):6d} archivos · {converted_bytes / 1e9:.2f} GB")

- **Decenas de miles de archivos chicos.** Los discos, la red y el Hub los
  manejan muy mal: copiar, subir o listar se vuelve lento.
- **Un JPEG por cuadro** repite casi la misma imagen 30 veces por segundo. El
  video aprovecha justamente eso: guarda las diferencias entre cuadros.
- **Para entrenar se lee al azar** (cuadro 8.000, después el 120, después el
  15.300...). Con JSON hay que abrir y parsear un archivo entero para sacar una
  fila.

El formato v3.0 de LeRobot responde a eso:

- **Pocos archivos grandes**: muchos episodios juntos en cada archivo de datos
  (hasta ~100 MB) y de video (hasta ~500 MB).
- **Parquet** para los números: un formato de tablas por columnas, comprimido,
  que permite leer filas sueltas rápido.
- **Video AV1** para las imágenes.
- **Metadata** que dice dónde está cada episodio: en qué archivo, en qué filas y
  en qué segundos del video.

## 2. Recorrido archivo por archivo

In [ ]:
for path in sorted(ROOT.rglob("*")):
    if path.is_file():
        print(f"{path.stat().st_size / 1e6:8.2f} MB  {path.relative_to(ROOT)}")

### `meta/info.json`: la ficha técnica

In [ ]:
info = json.loads((ROOT / "meta" / "info.json").read_text())
{k: v for k, v in info.items() if k != "features"}

- `codebase_version`: la versión del formato (v3.0). Datasets viejos del Hub
  pueden estar en v2.1, que tenía un archivo por episodio.
- `data_path` y `video_path`: plantillas para encontrar los archivos.
- `chunks_size`, `*_files_size_in_mb`: cuándo se empieza un archivo nuevo.
- `splits`: LeRobot no separa train/val acá; nuestro split vive aparte
  (`learning/act/split.json`).

Y las *features*: cada columna con su tipo, forma y nombres.

In [ ]:
for name, feature in info["features"].items():
    extra = f" · codec {feature['info']['video.codec']}" if feature["dtype"] == "video" else ""
    print(f"{name:36s} {feature['dtype']:8s} {str(feature['shape']):15s}{extra}")
print()
print("nombres de la acción:", info["features"]["action"]["names"][0][:4], "...")

Las cinco últimas (`timestamp`, `frame_index`, `episode_index`, `index`,
`task_index`) las agrega LeRobot solo. `index` es el número global del cuadro
en todo el dataset; `frame_index`, dentro de su episodio.

### `meta/episodes/`: dónde está cada episodio

In [ ]:
episodes = pd.concat(pd.read_parquet(p) for p in sorted((ROOT / "meta" / "episodes").rglob("*.parquet")))
location_columns = [c for c in episodes.columns if not c.startswith("stats/")]
episodes[location_columns].head(3).T

Para el episodio 0: sus filas van de `dataset_from_index` a `dataset_to_index`
en el archivo de datos `data/chunk-000/file-000`, y su video está entre los
segundos `from_timestamp` y `to_timestamp` del archivo de video. Así se
encuentra un cuadro cualquiera sin leer nada más.

Además, cada episodio trae **sus propias estadísticas** (las columnas
`stats/...`): las usamos en el notebook 03 para encontrar los episodios del
entrenamiento.

In [ ]:
stat_columns = [c for c in episodes.columns if c.startswith("stats/observation.state/")]
print(len([c for c in episodes.columns if c.startswith("stats/")]), "columnas de estadísticas, por ejemplo:")
print([c.split("/")[-1] for c in stat_columns])

### `meta/stats.json` y `meta/tasks.parquet`

`stats.json` es lo mismo pero para todo el dataset: de ahí sale la
normalización. `tasks.parquet` guarda las instrucciones en texto (las que
leerán los VLAs).

In [ ]:
stats = json.loads((ROOT / "meta" / "stats.json").read_text())
print("estadísticas de la acción:", list(stats["action"].keys()))
pd.read_parquet(ROOT / "meta" / "tasks.parquet")

### `data/`: los números

Una fila por cuadro. Se puede leer con pandas directo, sin LeRobot, y es la
forma más rápida de analizar trayectorias (así trabaja `toolkit.validation`).

In [ ]:
frames = pd.read_parquet(ROOT / "data" / "chunk-000" / "file-000.parquet")
print(frames.shape, "→ una fila por cuadro")
frames.head(3)

### `videos/`: las imágenes

Un solo mp4 con todos los episodios uno detrás del otro.

In [ ]:
import av

video = next((ROOT / "videos").rglob("*.mp4"))
with av.open(str(video)) as container:
    stream = container.streams.video[0]
    print(f"{stream.codec_context.name} · {stream.width}×{stream.height} · {float(stream.average_rate):.0f} fps · {float(stream.duration * stream.time_base) / 60:.1f} min")

## 3. `LeRobotDataset`: la forma normal de leerlo

Junta todo lo anterior y te devuelve un cuadro como diccionario de tensores,
decodificando del video la imagen que corresponde.

In [ ]:
dataset = LeRobotDataset("local/pick_mameluco_act_v1", root=ROOT, video_backend="pyav")
print(len(dataset), "cuadros ·", dataset.num_episodes, "episodios ·", dataset.fps, "fps")
item = dataset[500]
{k: (tuple(v.shape), str(v.dtype)) if hasattr(v, "shape") else v for k, v in item.items()}

Lo que conviene saber:

- `dataset.meta`: toda la metadata (features, estadísticas, episodios) sin
  tocar los datos.
- `dataset.hf_dataset`: la tabla de números, sin imágenes (rápida).
- `video_backend`: quién decodifica. En la Mac usamos `pyav` (ver toolkit).
- **Las imágenes vuelven en float entre 0 y 1, canales primero** (3, alto,
  ancho), aunque en el video sean bytes de 0 a 255 en (alto, ancho, 3).

### Cuidado: `episodes=[...]` no hace lo que parece

`LeRobotDataset` acepta `episodes=[...]` para "cargar sólo algunos episodios".
Suena perfecto para separar entrenamiento de evaluación. Miremos qué devuelve:

In [ ]:
subset = LeRobotDataset("local/pick_mameluco_act_v1", root=ROOT, video_backend="pyav", episodes=[0, 10, 20])
present = sorted({int(e) for e in subset.hf_dataset["episode_index"]})
print("pedimos 3 episodios · num_episodes dice", subset.num_episodes)
print("pero len() da", len(subset), "cuadros, y adentro hay", len(present), "episodios distintos")

**Devuelve todo.** En esta versión (0.4.1), `episodes` sólo decide qué
archivos bajar del Hub; con un dataset local, o uno donde todos los episodios
comparten archivo (el nuestro: uno solo), no filtra filas. Lo verificamos en
el código: `load_hf_dataset` carga todos los parquet sin filtrar.

La consecuencia es seria: entrenar con `--dataset.episodes=[...]` para dejar
afuera la evaluación **entrenaría igual con todo**, sin avisar. Por eso:

- Para separar train y evaluación **partimos el dataset en dos datasets
  físicos** con `split_dataset` (sección 6).
- La evaluación offline (`toolkit.eval.offline`) va episodio por episodio
  usando `meta.episodes`, así que no la afecta.

(La primera ACT sí respetó su separación: se entrenó en otra máquina con otra
versión, y su error en episodios no vistos es 4,5 veces el de entrenamiento,
notebook 05. Si hubiera visto esos episodios, los errores serían parecidos.)

Y la metadata sola, sin cargar nada pesado (así leíamos el dataset de ALOHA en
el L1):

In [ ]:
meta = LeRobotDatasetMetadata("local/pick_mameluco_act_v1", root=ROOT)
print(meta.total_episodes, "episodios ·", meta.total_frames, "cuadros · cámaras:", meta.camera_keys)

## 4. Ventanas de tiempo: `delta_timestamps`

Lo más importante para entender cómo se alimenta una política. Cada política
pide, para cada clave, **qué instantes relativos** necesita (en segundos). Ya lo
vimos con ACT (48 acciones futuras) y Diffusion (2 observaciones, 16 acciones)
en el L1. Acá lo usamos a mano, pidiendo pasado y futuro:

In [ ]:
fps = dataset.fps
window = LeRobotDataset(
    "local/pick_mameluco_act_v1", root=ROOT, video_backend="pyav",
    delta_timestamps={
        CAMERA: [-1.0, -0.5, 0.0],                          # tres imágenes: hace 1 s, hace 0,5 s, ahora
        "observation.state": [k / fps for k in range(-15, 1)],  # el último medio segundo de estado
        "action": [k / fps for k in range(30)],             # el próximo segundo de acciones
    },
)
sample = window[2000]
{k: tuple(v.shape) for k, v in sample.items() if hasattr(v, "shape") and v.ndim > 0}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 2.6))
for ax, image, label in zip(axes, sample[CAMERA], ["−1,0 s", "−0,5 s", "ahora"]):
    ax.imshow(image.permute(1, 2, 0)); ax.set_title(label); ax.axis("off")
plt.tight_layout()

Fijate las claves `*_is_pad`: marcan qué posiciones de la ventana caen
**fuera del episodio** (antes del primer cuadro o después del último) y fueron
rellenadas repitiendo el borde. El entrenamiento no las cuenta.

In [ ]:
start_of_episode = int(window.meta.episodes[5]["dataset_from_index"]) + 5
edge = window[start_of_episode]
print("a 5 cuadros del inicio, posiciones de estado rellenadas:", int(edge["observation.state_is_pad"].sum()), "de 16")
print("imágenes rellenadas:", edge[f"{CAMERA}_is_pad"].tolist())

**Una regla que esto impone**: el dataset tiene que estar a una frecuencia
constante (30 fps), porque las ventanas se piden en segundos y se traducen a
cuadros. Si faltaran cuadros, la ventana mezclaría instantes equivocados sin
avisar (con `tolerance_s` LeRobot sólo controla que el timestamp pedido
exista).

## 5. Aumentación de imágenes

Con pocos datos, la red puede memorizar detalles exactos de los píxeles
(iluminación de ese día, una mancha en la estantería) en vez de aprender la
tarea. La **aumentación** muestra en cada lote una versión un poco alterada de
la imagen: más brillo, otro contraste, un leve corrimiento. La tarea es la
misma; los píxeles no.

Nuestra primera ACT se entrenó **sin** aumentación (L1). Así se ve lo que
LeRobot trae por defecto:

In [ ]:
from lerobot.datasets.transforms import ImageTransforms, ImageTransformsConfig

config = ImageTransformsConfig(enable=True)
print("transformaciones disponibles:", list(config.tfs.keys()), "· como máximo", config.max_num_transforms, "por imagen")
transform = ImageTransforms(config)
original = dataset[3000][CAMERA]
torch.manual_seed(0)
fig, axes = plt.subplots(2, 3, figsize=(14, 5))
axes[0, 0].imshow(original.permute(1, 2, 0)); axes[0, 0].set_title("original")
for ax in axes.ravel()[1:]:
    ax.imshow(transform(original).permute(1, 2, 0).clamp(0, 1)); ax.set_title("aumentada")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()

Para activarla al entrenar: `--dataset.image_transforms.enable=true`. Se
aplica sólo en entrenamiento, nunca al evaluar.

---

# Para profundizar

## 6. Editar datasets

Todo se puede hacer con `lerobot-edit-dataset` en la terminal o con
`lerobot.datasets.dataset_tools` en Python. Las operaciones de esta versión:
borrar episodios, partir, unir y sacar una feature. Siempre crean un dataset
**nuevo**; el original no se toca.

Probamos con el chiquito de 3 episodios (cada operación tarda unos segundos
porque re-codifica el video).

In [ ]:
import shutil
from lerobot.datasets.dataset_tools import delete_episodes, split_dataset

demo = LeRobotDataset("local/nb_demo", root=DEMO_ROOT, video_backend="pyav")

out = DATASETS_DIR / "local" / "nb_demo_without_1"
shutil.rmtree(out, ignore_errors=True)
pruned = delete_episodes(demo, episode_indices=[1], output_dir=out, repo_id="local/nb_demo_without_1")
print("sin el episodio 1:", pruned.meta.total_episodes, "episodios,", pruned.meta.total_frames, "cuadros (antes", demo.meta.total_frames, ")")

out = DATASETS_DIR / "local" / "nb_demo_split"
shutil.rmtree(out, ignore_errors=True)
parts = split_dataset(demo, splits={"train": [0, 1], "val": [2]}, output_dir=out)
{name: part.meta.total_episodes for name, part in parts.items()}

Esto es lo que habría que hacer en vez de reconvertir para sacar los 4
episodios descartados: `delete_episodes` sobre el dataset de 65. Las
estadísticas del dataset nuevo se recalculan con lo que queda.

En Python:

```python
source = LeRobotDataset("local/pick_mameluco", root=DATASETS_DIR / "local" / "pick_mameluco")
delete_episodes(source, episode_indices=[0, 7, 20, 50],
                output_dir=DATASETS_DIR / "local" / "pick_mameluco_clean", repo_id="local/pick_mameluco_clean")
```

Existe el comando `lerobot-edit-dataset`, pero con datasets locales tiene una
rareza en esta versión: carga el original desde `--root` y guarda el
resultado en `--root/<new_repo_id>`, o sea **adentro** de la carpeta del
original (está en `lerobot/scripts/lerobot_edit_dataset.py`,
`get_output_path`). Con la API de Python elegís la carpeta de salida y listo.

Otras herramientas útiles:

- `lerobot-dataset-viz`: un episodio en el visor Rerun, con video y curvas
  sincronizados.
- `dataset.push_to_hub(private=True)`: subir al Hub (el nuestro, siempre
  privado: muestra el lugar real).

## 7. Detalles de bajo nivel

- **Episodios que no entran en un archivo**: cuando un archivo de datos pasa
  ~100 MB (o uno de video ~500 MB), LeRobot empieza `file-001`. Cada 1000
  archivos, una carpeta `chunk-001`. Nuestro dataset entra en uno solo de cada.
- **Cuantiles**: las estadísticas traen `q01`…`q99`. Hay políticas (π0.5) que
  normalizan con cuantiles en vez de media y desvío (notebook L4). En esta
  versión los cuantiles del dataset entero son un promedio de los de cada
  episodio: una aproximación.
- **Videos y precisión**: la imagen decodificada no es idéntica al JPEG
  original (notebook 02: ~3 niveles de 255 por píxel). Si una tarea dependiera
  de detalles finos, se puede guardar como imágenes en vez de video.

## Referencias

- Formato v3.0: [documentación de LeRobot](https://huggingface.co/docs/lerobot/lerobot-dataset-v3)
- Herramientas de datasets: [using_dataset_tools](https://huggingface.co/docs/lerobot/using_dataset_tools)
- Código: `lerobot/datasets/lerobot_dataset.py` y `dataset_tools.py` (dentro de `.venv`).